# 01a — ID check

Prepares the Ministry of Commerce (MC) active commercial registrations for the survival model.

**Rules for this notebook**
- The raw files are never changed. Anything cleaned is saved as a new file.
- Nothing is dropped. Problem rows get a flag column instead.
- Data never goes into Git. Clear all outputs before committing.

**The preprocessing notebooks, in order** (each one loads what the one before saved):
1. `01a_id_check`: did the ID change, or only its column name? (stands alone)
2. `01b_all_quarters`: reads the 30 Excel files (2023 Q1 – 2026 Q3) once and saves `mc_all_quarters.parquet`
3. `01c_id_matching`: one ID per business, saves `mc_matched.parquet`
4. `01d_survival_label`: the one-year label, saves `labels.parquet`
5. `01e_ssic_classification`: SSIC from the name, saves `ssic_by_business.csv`, `business_names.csv`, `business_city.csv`
6. `01f_old_data_branches`: old data (2023 Q1 – 2025 Q2), every branch joined to its main (stands alone)

## Part 1 — Did the ID change, or only its column name?

Up to 2025 Q3 the ID column is called **الرقم الوطني**. From 2025 Q4 it is called **الرقم الموحد**.
This part checks whether the numbers stayed the same, by comparing **2025 Q2** (the last complete quarter
with the old name) with **2025 Q4** (the first quarter with the new name).
2025 Q3 is skipped on purpose: most of its rows are missing and half of it has no ID.

### Cell 1 — Setup
Finds the data. On Colab it downloads the raw files from the team's Google Drive folder (temporary copies,
gone when the session ends). On a laptop it reads the local copy in `DATA_DIR`.

The results that the next notebook loads go in `SAVED_DIR`: the folder **saved** inside the team's shared Drive
folder **Darb DataSets**. Everyone loads the same results, and they are still there in a new Colab session.
On Colab: add a shortcut to Darb DataSets in your My Drive first (Drive → right-click the folder → Organize →
Add shortcut → My Drive). Colab asks once to connect Drive.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    !pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
    from google.colab import drive
    drive.mount('/content/drive')
    TEAM_DIR = Path('/content/drive/MyDrive/Darb DataSets')     # the team's shared folder (a shortcut in My Drive)
    if not TEAM_DIR.exists():
        raise FileNotFoundError('Darb DataSets is not in your My Drive: add a shortcut to it (see the text above), then run this cell again.')
    SAVED_DIR = TEAM_DIR / 'saved'                              # shared with the team, kept between Colab sessions
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', r'C:\Users\Nasser Alyahya\darb-data'))
    SAVED_DIR = DATA_DIR / 'saved'

MC_DIR = DATA_DIR / 'MC'
SAVED_DIR.mkdir(parents=True, exist_ok=True)
print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR, '| saved results:', SAVED_DIR)

### Cell 1b — Read an MC file (Excel only the first time)
Reading the big Excel files is the slow part. The first time a file is read, an **exact copy** of it (every row
and every column, all as text) is saved in `SAVED_DIR/mc_excel_copies`. After that the copy is read instead,
in seconds, and on Colab the Excel file is not downloaded again. Delete a copy to read its Excel file again.

In [ ]:
import pandas as pd

EXCEL_COPIES = SAVED_DIR / 'mc_excel_copies'
EXCEL_COPIES.mkdir(parents=True, exist_ok=True)

def has_copy(name):
    return (EXCEL_COPIES / f'{name}.parquet').exists()

def read_mc(name, usecols=None):
    # One MC file, all as text: from its saved copy if there is one, otherwise from Excel (and the copy is saved)
    copy = EXCEL_COPIES / f'{name}.parquet'
    if copy.exists():
        return pd.read_parquet(copy, columns=usecols)
    table = pd.read_excel(MC_DIR / name, engine='calamine', dtype=str)
    if all(isinstance(c, str) for c in table.columns):     # parquet needs text column titles
        temp = copy.with_suffix('.part')                     # a copy cut off halfway is never used
        table.to_parquet(temp, index=False)
        temp.rename(copy)
    return table[usecols] if usecols else table

### Cell 2 — The files this part needs
Only the 4 files of 2025 Q2 and 2025 Q4 (each quarter comes in two parts).
The IDs are the files' Google Drive IDs, used only when running on Colab.

In [ ]:
FILES = {
    '2025Q2': [('2025-02q-active-crs_PART1-2.xlsx', '1pAlWmlVTGwCY2XnwhuujvFz9w8ys3nfp'),
               ('2025-02q-active-crs_PART2.xlsx',   '14LNdY7UCE-itRV4ubyfRzapGtwgoDda_')],
    '2025Q4': [('2025-04q-active-crs_PART1.xlsx',   '1tzttjeDaz8WQrFSGE7WYI5AMQiNSsRrM'),
               ('2025-04q-active-crs_PART2.xlsx',   '1rqw-5VqvreNlA7hJN4sY3wOzgmOI485a')],
}

if IN_COLAB:
    import gdown
    MC_DIR.mkdir(parents=True, exist_ok=True)
    for parts in FILES.values():
        for name, drive_id in parts:
            if not (MC_DIR / name).exists() and not has_copy(name):
                gdown.download(id=drive_id, output=str(MC_DIR / name), quiet=True)

for quarter, parts in FILES.items():
    for name, _ in parts:
        path = MC_DIR / name
        print(quarter, name, 'found' if path.exists() else ('saved copy' if has_copy(name) else 'MISSING'), f'({path.stat().st_size / 1e6:.0f} MB)' if path.exists() else '')

### Cell 3 — Read only the columns the test needs
ID, name, start date and city. The two quarters use different column names, so each is renamed
to the same four names: `id`, `name`, `start_date`, `city`. Everything is read as text, so a number
like `7001234567` is never changed (no rounding, no lost zeros).

In [ ]:
import pandas as pd

COLUMNS = {
    # old names (up to 2025 Q3)
    '2025Q2': {'الرقم الوطني': 'id', 'اسم السجل التجاري': 'name',
               'تاريخ الإصدار (ميلادي)': 'start_date', 'اسم المدينة (بالعربية)': 'city'},
    # new names (from 2025 Q4)
    '2025Q4': {'الرقم الموحد': 'id', 'اسم السجل': 'name',
               'تاريخ انشاء السجل': 'start_date', 'المدينة': 'city'},
}

def read_quarter(quarter):
    tables = []
    for name, _ in FILES[quarter]:
        part = read_mc(name, usecols=list(COLUMNS[quarter]))
        tables.append(part.rename(columns=COLUMNS[quarter]).assign(source_file=name))
    table = pd.concat(tables, ignore_index=True)
    for column in ['id', 'name', 'city', 'start_date']:
        table[column] = table[column].str.strip()
    table['start_day'] = table['start_date'].str[:10]   # date only, without the time
    return table

q2 = read_quarter('2025Q2')
q4 = read_quarter('2025Q4')
print(f'2025 Q2: {len(q2):,} rows | 2025 Q4: {len(q4):,} rows')

### Cell 4 — Compare the IDs (numbers)
1. How many IDs each quarter has, and whether any ID repeats inside a quarter.
2. How many 2025 Q2 IDs also appear in 2025 Q4.
3. For the IDs found in both: do the name, start date and city also agree? If they do, it is the same business
   and only the column name changed.

In [ ]:
def describe(label, table):
    ids = table['id']
    print(f'{label}: {len(table):,} rows | empty ID: {ids.isna().sum():,} | '
          f'unique IDs: {ids.nunique():,} | IDs that repeat: {ids.duplicated(keep=False).sum():,} rows')
    print(f'   IDs starting with 7: {ids.str.startswith("7").mean():.1%} | 10 digits long: {ids.str.fullmatch(r"\d{10}").mean():.1%}')

describe('2025 Q2', q2)
describe('2025 Q4', q4)

q2_ids, q4_ids = set(q2['id'].dropna()), set(q4['id'].dropna())
both = q2_ids & q4_ids
print()
print(f'IDs in both quarters: {len(both):,}')
print(f'   = {len(both) / len(q2_ids):.1%} of the 2025 Q2 IDs')
print(f'   = {len(both) / len(q4_ids):.1%} of the 2025 Q4 IDs')

# For the shared IDs, compare the details (first row of each ID, to keep it simple)
pairs = (q2.drop_duplicates('id').set_index('id')[['name', 'start_day', 'city']]
         .join(q4.drop_duplicates('id').set_index('id')[['name', 'start_day', 'city']],
               how='inner', lsuffix='_q2', rsuffix='_q4'))
print()
print('For the IDs in both quarters, how often the details agree:')
for column in ['name', 'start_day', 'city']:
    print(f'   same {column:<10}: {(pairs[column + "_q2"] == pairs[column + "_q4"]).mean():.1%}')


### Cell 5 — Twenty random examples
Twenty IDs found in both quarters, side by side (2025 Q2 on the left, 2025 Q4 on the right),
to look at with our own eyes.

In [ ]:
pairs.sample(20, random_state=1)

## Part 1b — Same business, different ID?

### Cell 6 — Missing businesses that may have come back under a new ID
About 109,000 businesses of 2025 Q2 are not in 2025 Q4. Most probably closed, but some may have
re-registered with a **new** ID (for example a مؤسسة that became a شركة). If we do not catch them, they
would wrongly count as businesses that did not survive.

This cell only **counts and shows examples**. It links nothing. Two ways to look for a possible new ID:
- **A (strict):** a new 2025 Q4 ID with the same name, start date and city.
- **B (looser):** a new 2025 Q4 ID with the same name once the legal form (شركة، مؤسسة، المحدودة…) is removed,
  in the same city. A re-registered business often gets a new start date, so B does not require the same date.

"More than one candidate" means a match is unsure: those must never be linked automatically.

In [ ]:
import re

# Businesses in only one of the two quarters
missing = q2[~q2['id'].isin(q4_ids)].drop_duplicates('id')    # in 2025 Q2, not in 2025 Q4
new_q4  = q4[~q4['id'].isin(q2_ids)]                           # in 2025 Q4, not in 2025 Q2
print(f'In 2025 Q2 but not in 2025 Q4: {len(missing):,}')
print(f'In 2025 Q4 but not in 2025 Q2: {len(new_q4):,}')

def core_name(name):
    """The name without its legal form and spelling differences, for comparing only:
    'شركة الوليد صالح الحربي المحدودة' and 'مؤسسة الوليد صالح الحربي' both become 'الوليد صالح الحربي'."""
    t = str(name)
    t = re.sub('[إأآ]', 'ا', t).replace('ة', 'ه').replace('ى', 'ي')
    t = re.sub(r'\b(شركه|مؤسسه|المحدوده|محدوده|ذات مسؤوليه|شخص واحد|مساهمه|مبسطه|فرع)\b', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

missing = missing.assign(core=missing['name'].map(core_name))
new_q4  = new_q4.assign(core=new_q4['name'].map(core_name))

def candidates(keys, label):
    found = missing.merge(new_q4, on=keys, suffixes=('_q2', '_q4'))
    per_business = found.groupby('id_q2')['id_q4'].nunique()
    print(f'{label}:')
    print(f'   missing Q2 businesses with a possible new ID: {len(per_business):,} '
          f'(exactly one candidate: {(per_business == 1).sum():,} | more than one: {(per_business > 1).sum():,})')
    return found

print()
strict = candidates(['name', 'start_day', 'city'], 'A. Same name + same start date + same city')
loose  = candidates(['core', 'city'], 'B. Same name without the legal form + same city (any start date)')

# Examples to look at with our own eyes
show = ['id_q2', 'name_q2', 'start_day_q2', 'id_q4', 'name_q4', 'start_day_q4', 'city']
print()
print('Examples of B (10 random):')
loose[show].sample(min(10, len(loose)), random_state=1)


### Cell 7 — The ID that repeats in 2025 Q2
The two rows that share one ID, to see if they are a duplicate or two different records.
Nothing is deleted: a duplicate will be flagged, not dropped.

In [ ]:
repeated = q2[q2['id'].duplicated(keep=False)]
print(f'Rows whose ID repeats in 2025 Q2: {len(repeated)}')
repeated[['id', 'name', 'start_date', 'city', 'source_file']]
